# 05. Layer 2 models

Layer 2 has two steps (decided in `specs/layer2-impact.md`):

1. **Rainfall:** ONI (this and last quarter) predicts rainfall percent of normal for each province and quarter. This feeds the map colours.
2. **Yield:** rainfall percent and ONI (this and last quarter) predict yield change against the normal (a straight line through the 10 previous years), clipped to -100 .. +100 percent.

Every model in a step is scored on the same rows, with MAE and RMSE (lower is better). A reference row with no model ("always normal": 100 percent rainfall, 0 percent yield change) shows what a model must beat.

The code lives in `backend/app/layer2/`. This notebook only runs it and shows the results.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../backend").resolve()))  # so `app` can be imported

from app.layer1_forecast.labels import find_events
from app.layer2.features import RAIN_TARGET, YIELD_TARGET, load_tables, split_by_year
from app.layer2.train import run_comparison

tables = load_tables()
for name, table in tables.items():
    for part, rows in split_by_year(table).items():
        print(f"{name:5s} {part:10s} {len(rows):6d} rows, {rows['year'].min()} to {rows['year'].max()}")

rain  train        9072 rows, 1981 to 2008
rain  validation   2592 rows, 2009 to 2016
rain  test         3159 rows, 2017 to 2026
yield train       16073 rows, 1997 to 2008
yield validation  12855 rows, 2009 to 2016
yield test        16434 rows, 2017 to 2026


The yield table really starts in **1997**, not 1987: the normal needs 10 earlier years with a yield. So the yield models train on 1997 to 2008. The rainfall models train on 1981 to 2008 (decided), which includes the strong 1982-83 El Nino.

## 1. Fit, choose and test

`run_comparison` fits every model on the training years and scores it on validation. It chooses the lowest validation RMSE (the reference cannot win), then scores every model once on the test years, fitted on everything up to 2016. This takes about a minute.

In [2]:
results = run_comparison(tables)

for step in ["rain", "yield"]:
    part = results[step]
    print(f"Step {step}: validation")
    print(part["validation"].round(3).to_string(index=False))
    print(f"Chosen: {part['best']}")
    print(f"Step {step}: test (fitted up to 2016)")
    print(part["test"].round(3).to_string(index=False))
    print()

Step rain: validation
           model  rows    mae   rmse
reference_normal  2592 30.715 41.528
   province_line  2592 25.969 36.459
   random_forest  2592 25.532 34.947
         xgboost  2592 23.982 33.884
Chosen: xgboost
Step rain: test (fitted up to 2016)
           model  rows    mae   rmse
reference_normal  3159 27.638 36.149
   province_line  3159 21.934 30.510
   random_forest  3159 25.862 35.259
         xgboost  3159 23.128 31.834

Step yield: validation
            model  rows    mae   rmse
 reference_normal 12855 11.750 17.075
linear_regression 12855 14.425 19.481
    random_forest 12855 15.012 20.679
          xgboost 12855 14.376 19.344
   departure_line 12855 11.773 16.980
Chosen: departure_line
Step yield: test (fitted up to 2016)
            model  rows    mae   rmse
 reference_normal 16434 10.788 16.765
linear_regression 16434 11.642 17.164
    random_forest 16434 12.237 17.800
          xgboost 16434 11.826 17.272
   departure_line 16434 10.853 16.777



## 2. The yield model as the app uses it

The yield model is trained on observed rainfall. In the app it gets step 1's *predicted* rainfall instead, because future rainfall is unknown. This is the honest score for the app.

In [3]:
print(results["chained"].round(3).to_string(index=False))

    period          model  rows    mae  rmse
validation departure_line 12855 11.877 17.02
      test departure_line 16434 10.833 16.73


## 3. Lag check: does last quarter matter?

The chosen yield model is fitted twice on the training years: with this and last quarter's rainfall and ONI, and with this quarter's only. A lower validation RMSE with both means last quarter helps.

In [4]:
print(results["lag_check"].round(3).to_string(index=False))

               inputs          model  rows    mae   rmse
this and last quarter departure_line 12855 11.773 16.980
    this quarter only departure_line 12855 11.826 17.117


## 4. El Nino against neutral quarters

Same quarter rule as notebook 04: a quarter is El Nino when at least 2 of its 3 months are inside an El Nino event (five-season rule, `find_events`), La Nina the same way, and neutral when it touches no event. Mixed quarters are left out.

For each period, the chosen models predict with the fit used for that period's score (validation: fitted on train; test: fitted up to 2016). The table compares the median predicted and actual values.

In [5]:
enso = pd.read_csv("../data/processed/enso_monthly.csv", parse_dates=["date"])[["date", "oni"]].dropna()
last_month = enso["date"].max()


def months_by_event(events):
    """Every month inside an event."""
    out = set()
    for e in events.itertuples():
        end = e.end_month if pd.notna(e.end_month) else last_month  # ongoing event
        out.update(pd.date_range(e.start_month, end, freq="MS"))
    return out


el_nino_months = months_by_event(find_events(enso))
la_nina_months = months_by_event(find_events(enso.assign(oni=-enso["oni"])))  # mirrored rule

months = pd.DataFrame({"date": pd.date_range("1981-01-01", "2026-12-01", freq="MS")})
months["year"], months["quarter"] = months["date"].dt.year, months["date"].dt.quarter
months["el_nino"] = months["date"].isin(el_nino_months)
months["la_nina"] = months["date"].isin(la_nina_months)
phase = months.groupby(["year", "quarter"], as_index=False)[["el_nino", "la_nina"]].sum()
phase["phase"] = np.select(
    [phase["el_nino"] >= 2, phase["la_nina"] >= 2, (phase["el_nino"] == 0) & (phase["la_nina"] == 0)],
    ["El Nino", "La Nina", "Neutral"], default="Mixed")

rows = []
for step, target in [("rain", RAIN_TARGET), ("yield", YIELD_TARGET)]:
    part = results[step]
    for period, model in [("validation", part["validation_model"]), ("test", part["test_model"])]:
        scored = part["splits"][period].assign(predicted=lambda d, m=model: m.predict(d).astype(float))
        scored = scored.merge(phase[["year", "quarter", "phase"]], on=["year", "quarter"], how="left")
        scored = scored[scored["phase"].isin(["El Nino", "Neutral"])]
        summary = scored.groupby("phase").agg(rows=(target, "size"), actual=(target, "median"),
                                              predicted=("predicted", "median"))
        rows.append(summary.assign(step=step, period=period).reset_index())

effect = pd.concat(rows, ignore_index=True)[["step", "period", "phase", "rows", "actual", "predicted"]]
print(effect.round(1).to_string(index=False))

 step     period   phase  rows  actual  predicted
 rain validation El Nino   810    74.7       70.0
 rain validation Neutral  1053   107.7       96.3
 rain       test El Nino   810    74.3       71.8
 rain       test Neutral  1053   114.0       94.7
yield validation El Nino  4078    -2.9       -2.3
yield validation Neutral  5255    -1.1        0.2
yield       test El Nino  4231    -2.2       -0.6
yield       test Neutral  5427     0.1       -1.2


## 5. Two more yield candidates (validation only)

The models above learned a +4 percent level from 1997 to 2008 and lose to "always normal". Two simpler candidates are tried on the **same validation rows**. No test score is computed for them here.

- **A. "Normal rain = 0 percent" line.** Linear regression with **no intercept** (no fixed starting value) on departures from normal: rain percent minus 100 (this and last quarter) and ONI (this and last quarter; ONI's normal is 0). A quarter with normal rain and ONI 0 always predicts 0 percent, so the model cannot learn a level.
- **B. Past El Nino average (no ML).** For each crop type, province and quarter, the median yield change over its El Nino quarters in the training years. It predicts that median in El Nino quarters (0 if the cell had none in training) and 0 in every other quarter.

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

yield_split = split_by_year(tables["yield"])
train_y, valid_y = yield_split["train"], yield_split["validation"]
assert len(train_y) == 16073 and len(valid_y) == 12855, "not the same rows as the models above"


def departures(d):
    """Departures from normal: rain percent minus 100, ONI as it is (its normal is 0)."""
    return pd.DataFrame({"rain": d["rain_pct"] - 100, "rain_prev": d["rain_pct_prev"] - 100,
                         "oni": d["oni"], "oni_prev": d["oni_prev"]})


# A: no intercept, so normal conditions always predict 0 percent.
line_a = LinearRegression(fit_intercept=False).fit(departures(train_y), train_y[YIELD_TARGET])
pred_a = line_a.predict(departures(valid_y))
coef = dict(zip(departures(train_y).columns, line_a.coef_))
print("A coefficients (yield change in percent):")
print(f"  per 10 points of rain below normal, this quarter: {-10 * coef['rain']:+.2f}")
print(f"  per 10 points of rain below normal, last quarter: {-10 * coef['rain_prev']:+.2f}")
print(f"  per +1 ONI, this quarter: {coef['oni']:+.2f}")
print(f"  per +1 ONI, last quarter: {coef['oni_prev']:+.2f}")

# B: median yield change in each cell's training El Nino quarters.
cell = ["crop_type", "province", "quarter"]
with_phase_train = train_y.merge(phase[["year", "quarter", "phase"]], on=["year", "quarter"], how="left")
el_nino_train = with_phase_train[with_phase_train["phase"] == "El Nino"]
cell_median = el_nino_train.groupby(cell)[YIELD_TARGET].median().rename("el_nino_median")
cell_count = el_nino_train.groupby(cell).size()

all_cells = tables["yield"][cell].drop_duplicates()
counts = all_cells.merge(cell_count.rename("n").reset_index(), on=cell, how="left")["n"].fillna(0)
print()
print("B: cells by number of training El Nino quarters")
print(pd.cut(counts, [-1, 0, 1, 2, np.inf], labels=["0", "1", "2", "3+"]).value_counts().sort_index().to_string())

valid_phase = valid_y.merge(phase[["year", "quarter", "phase"]], on=["year", "quarter"], how="left")
assert len(valid_phase) == len(valid_y)
valid_phase = valid_phase.merge(cell_median.reset_index(), on=cell, how="left")
pred_b = np.where(valid_phase["phase"] == "El Nino", valid_phase["el_nino_median"].fillna(0), 0.0)

actual = valid_y[YIELD_TARGET]
new_rows = pd.DataFrame([
    {"model": name, "rows": len(actual), "mae": mean_absolute_error(actual, p),
     "rmse": float(np.sqrt(mean_squared_error(actual, p)))}
    for name, p in [("A_no_intercept_line", pred_a), ("B_past_el_nino_median", pred_b)]])
validation_table = pd.concat([results["yield"]["validation"], new_rows], ignore_index=True)
print()
print("Yield, validation (same rows for every model):")
print(validation_table.round(2).to_string(index=False))

medians = valid_phase.assign(A=pred_a, B=pred_b)
medians = medians[medians["phase"].isin(["El Nino", "Neutral"])]
print()
print("Median yield change on validation, actual against predicted:")
print(medians.groupby("phase").agg(rows=(YIELD_TARGET, "size"), actual=(YIELD_TARGET, "median"),
                                   A=("A", "median"), B=("B", "median")).round(2).to_string())

A coefficients (yield change in percent):

  per 10 points of rain below normal, this quarter: +0.29
  per 10 points of rain below normal, last quarter: +0.24
  per +1 ONI, this quarter: +1.01
  per +1 ONI, last quarter: -5.22

B: cells by number of training El Nino quarters
n
0     353
1     399
2     112
3+    937



Yield, validation (same rows for every model):
                model  rows   mae  rmse
     reference_normal 12855 11.75 17.07
    linear_regression 12855 14.42 19.48
        random_forest 12855 15.01 20.68
              xgboost 12855 14.38 19.34
       departure_line 12855 11.77 16.98
  A_no_intercept_line 12855 11.77 16.98
B_past_el_nino_median 12855 13.20 19.35

Median yield change on validation, actual against predicted:
         rows  actual     A     B
phase                            
El Nino  4078   -2.93 -2.33  0.41
Neutral  5255   -1.09  0.21  0.00


## 6. Findings

**Rows.** Rainfall: train 9,072 (1981 to 2008), validation 2,592, test 3,159. Yield: train 16,073 (1997 to 2008, because the normal needs 10 earlier years), validation 12,855, test 16,434.

**Step 1, rainfall.** Every model beats "always normal".
- Validation RMSE: reference 41.5, province line 36.5, Random Forest 34.9, XGBoost 33.9. XGBoost is chosen.
- Test RMSE (fitted up to 2016): reference 36.1, province line 30.5, Random Forest 35.3, XGBoost 31.8. On the test years the simple province line does best; the choice was made on validation, so it stays XGBoost.
- El Nino quarters: actual median rainfall about 74 to 75 percent of normal, predicted 70 to 72. Neutral quarters: actual 108 to 114, predicted 95 to 96. The model sees the El Nino drop clearly.

**Step 2, yield.** Candidate A, the no-intercept line on departures from normal (section 5), is chosen.
- Validation RMSE: reference 17.08, linear 19.5, Random Forest 20.7, XGBoost 19.3, A 16.98. A is the only model that beats "always normal" on validation.
- Test RMSE (fitted up to 2016): reference 16.77, A 16.78. On the test years A is level with "always normal", not better.
- With step 1's predicted rainfall (what the app does): A scores validation 17.02, test 16.73.
- Linear regression, Random Forest and XGBoost lost because they learned a fixed level of about +4 percent from 1997 to 2008, when yields sat above their normal. A has no intercept, so a normal quarter always predicts 0.
- The El Nino effect is real but small: about 5 percent lower yield per +1 rise in last quarter's ONI.

**Lag check (A, validation RMSE).** This and last quarter 16.98; this quarter only 17.12. Last quarter helps a little, so both are kept.

**What this means.** Step 1 works and gives the map colours. Step 2 shows the size of the El Nino effect on yield, but it is no more accurate than "always normal" for a single quarter on the test years. This is reported honestly.

**Two more yield candidates (section 5, validation only, same 12,855 rows).**
- A, no-intercept line on departures from normal: RMSE 16.98, MAE 11.77, against the reference 17.08 and 11.75. It is the only model that beats "always normal" on RMSE, by 0.09; on MAE it is level with the reference. Median prediction: El Nino -2.3 (actual -2.9), neutral +0.2 (actual -1.1).
- A's weights: +1 ONI last quarter lowers yield by 5.2 percent; +1 ONI this quarter raises it by 1.0; rain matters very little (10 points of rain below normal changes yield by about +0.3, the wrong sign for drought).
- B, past El Nino median per cell: RMSE 19.35, worse than the reference. Of 1,801 cells, 353 had no training El Nino quarter and 399 had only one, so the medians are noisy.